In [1]:
import pymongo
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, date
import os
import pprint

load_dotenv(override=True)

True

### setup connection

In [2]:
mongo_uri = os.getenv("mongo_uri")
# mongo_uri_local = os.getenv("mongo_uri_local")

try:
    client = pymongo.MongoClient(mongo_uri)
except NameError:
    print("mongo_uri not available, trying mongo_uri_local")
    client=pymongo.MongoClient(mongo_uri_local)

from pymongo.uri_parser import parse_uri
# print(parse_uri(mongo_uri_local)["options"])
# print(repr(mongo_uri_local))



client.list_database_names()
db = client["klapp-prod"]

# Data ideas

## 1. Students with duplicated parents (parents share the same name)

For each student, look up their linked parents and flag the student if two *different* parent accounts (different `_id`) resolve to the same name.

In [9]:
list(db["user"].find({"role": "parent"}).limit(1))

[{'_id': ObjectId('598c4487c9b1fb417ce4fdc3'),
  'email': 'nikarofi@gmail.com',
  'password': '$2a$10$3fTyo/BI8PAql9FA8Xll2.ONBxvBBFM8x977bsqcecfhqPLzTcbQq',
  'role': ['parent'],
  'first_name': 'Nicole',
  'last_name': 'Kaspar',
  'addresses': [],
  'telephone_numbers': [''],
  'accepted_class_links': [ObjectId('599beb52c9b1fb1e7c4fcb8c'),
   ObjectId('599bf77dc9b1fb1e79401075'),
   ObjectId('5ac25486c9b1fb24d94b12fa'),
   ObjectId('5b6bda8cdf33610034294c5c'),
   ObjectId('5b6bda8bdf336100406f48d7')],
  'rejected_class_links': [ObjectId('5991f840c9b1fb1e7e9c0cd6')],
  'communication_channels': [{'school': ObjectId('598c7998c9b1fb1c74918d5e'),
    'communication_channels': []},
   {'school': ObjectId('5ee85b8d31501c003f2391bc'),
    'communication_channels': []},
   {'school': ObjectId('6544e4a8a2572401a7524694'),
    'communication_channels': []}],
  'devices': [],
  'sms_number': '+41795779349',
  'confirmed': True,
  'confirmation_token': 'eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJhY

In [10]:
db["student"].find_one()

{'_id': ObjectId('599fce66c9b1fb7f60f7d0f5'),
 'school': ObjectId('598c7998c9b1fb1c74918d5e'),
 'parents': [ObjectId('599fcceac9b1fb7f5ed37186'),
  ObjectId('59aefa1fc9b1fb078f665824')],
 'first_name': 'Diego',
 'last_name': 'Enderlin',
 'birth_date': datetime.datetime(2011, 11, 21, 0, 0),
 'code_parent': 'P-T5GC48',
 'code_student': 'S-3QQANH',
 'lo_student_id': 'MO67OPJVL1-STUDENT-14048',
 'user': None,
 'deleted_at': datetime.datetime(2026, 8, 10, 11, 29, 29, 962000),
 'updated_at': datetime.datetime(2026, 8, 10, 11, 29, 29, 962000),
 'external': {'lehreroffice': {'id': '700',
   'external_id': '14048',
   'school_year': '2023/24'}},
 'search': {'concat_first_and_last_name': 'Diego Enderlin',
  'concat_last_and_first_name': 'Enderlin Diego'},
 'updatedAt': datetime.datetime(2023, 11, 23, 9, 35, 9, 722000),
 'meta': {'jokerdays': 4}}

In [13]:
pipeline = [
    # no role filter needed - the "student" collection only contains students
    {"$lookup": {
        "from": "user",
        "localField": "parents",
        "foreignField": "_id",
        "as": "parent_docs",
    }},
    {"$addFields": {
        "parent_first_name": "$parent_docs.first_name",
        "parent_last_name": "$parent_docs.last_name",
    }},
    {"$match": {
        # flag when there are more parent docs than distinct (first_name, last_name) pairs
        "$expr": {
            "$gt": [
                {"$size": "$parent_docs"},
                {"$size": {"$setUnion": [
                    {"$map": {
                        "input": "$parent_docs",
                        "as": "p",
                        "in": {"$concat": ["$$p.first_name", " ", "$$p.last_name"]},
                    }},
                    [],
                ]}},
            ]
        }
    }},
    {"$project": {
        "first_name": 1,
        "last_name": 1,
        "parent_docs._id": 1,
        "parent_first_name": 1,
        "parent_last_name": 1,
    }},
]

df = pd.DataFrame(list(db["student"].aggregate(pipeline)))

In [14]:
df

,_id,first_name,last_name,parent_docs,parent_first_name,parent_last_name
0,5b68a6c0df336100406f4898,Lucas,Nigg,"[{'_id': 5ba62eed4b3a4c0039b54d0b}, {'_id': 61...","[Martin, Sara, Sara]","[Nigg, Nigg, Nigg]"
1,5b6bdb5adf33610034294c72,Ozan,Yücel,"[{'_id': 69e90232f14a2be4004dbeac}, {'_id': 5b...","[Dogukan, Dogukan]","[Yücel, Yücel]"
2,5b7c31571037ae003b3bb331,Mattia,Andres,"[{'_id': 5b8bbb669fd0670029b093bd}, {'_id': 5b...","[Franziska, Franziska, Thomas]","[Andres, Andres, Griess]"
3,5b7c313f1037ae003a91d571,Paolo,Andres,"[{'_id': 5b8bbac29fd067002be131b6}, {'_id': 5b...","[Thomas, Franziska, Franziska]","[Griess, Andres, Andres]"
4,5c5d2217862d1c0063525770,Jiyan,Ekinci,"[{'_id': 5d09003204b6dd0068957ea4}, {'_id': 69...","[Sevim, Sevim]","[Ekinci, Ekinci]"
...,...,...,...,...,...,...
6830,6aa923dba881067eb967025d,Luan,Seferi,"[{'_id': 6a97ffd17bc3eb42f32df5e0}, {'_id': 6a...","[Januz, Fazile, Seferi, Fazile]","[Seferi, Seferi, Fazile, Seferi]"
6831,6aa9255bc6bbf071c9a1016d,Luan,Seferi,"[{'_id': 6a9aee86a53aab6cb1f0370f}, {'_id': 6a...","[Fazile, Januz, Seferi, Fazile]","[Seferi, Seferi, Fazile, Seferi]"
6832,6aa9255bc6bbf071c9a1017f,Arianna,Ramaj,"[{'_id': 6782b9aa73653d032a14bf05}, {'_id': 67...","[Joakim, Albulene, Albulene]","[Ramaj, Ballabani, Ballabani]"
6833,6aa927b3b9b37774f579372c,Luan,Seferi,"[{'_id': 6a97ffd17bc3eb42f32df5e0}, {'_id': 6a...","[Januz, Fazile, Seferi, Fazile]","[Seferi, Seferi, Fazile, Seferi]"


In [15]:
pipeline = [
    {"$lookup": {
        "from": "user",
        "localField": "parents",
        "foreignField": "_id",
        "as": "parent_docs",
    }},
    {"$match": {
        "$expr": {
            "$gt": [
                {"$size": "$parent_docs"},
                {"$size": {"$setUnion": [
                    {"$map": {
                        "input": "$parent_docs",
                        "as": "p",
                        "in": {"$concat": ["$$p.first_name", " ", "$$p.last_name"]},
                    }},
                    [],
                ]}},
            ]
        }
    }},
    {"$group": {
        # siblings from the same family share the same "parents" array, so sorting+grouping
        # on it collapses them into a single incident instead of one per child
        "_id": {"$sortArray": {"input": "$parents", "sortBy": 1}},  # requires MongoDB 5.2+
        "children": {"$push": {"_id": "$_id", "first_name": "$first_name", "last_name": "$last_name"}},
        "num_children": {"$sum": 1},
    }},
]

results = list(db["student"].aggregate(pipeline))
print(f"distinct duplicate-parent incidents: {len(results)}")
print(f"students affected (sum of num_children): {sum(r['num_children'] for r in results)}")
results[:10]

distinct duplicate-parent incidents: 4279
students affected (sum of num_children): 6835


[{'_id': [ObjectId('68a74582ee0913b1227e89bd'),
   ObjectId('69ef66007569f5be1bac9ba0')],
  'children': [{'_id': ObjectId('6889e6e2a0055cc3371ad0a0'),
    'first_name': 'Dominik',
    'last_name': 'Ediae'}],
  'num_children': 1},
 {'_id': [ObjectId('6130eb5b29fe570d569336eb'),
   ObjectId('613f1ba6a8a7b30d7134e866'),
   ObjectId('69c6a0d018e60b0bb1c42f3f')],
  'children': [{'_id': ObjectId('64b5105fa5bf9d01a81e9964'),
    'first_name': 'Doa',
    'last_name': 'Kasami'}],
  'num_children': 1},
 {'_id': [ObjectId('6360e9aa673d1110cdf30595'),
   ObjectId('666843b40f9414dace6c7cc6'),
   ObjectId('666d7cc507a298cbf20d74f1')],
  'children': [{'_id': ObjectId('65e9e3673dab3206003bb563'),
    'first_name': 'Léanne',
    'last_name': 'Möschberger'}],
  'num_children': 1},
 {'_id': [ObjectId('671b863a88107c35045d2a55'),
   ObjectId('6a0214060676ad010302fa5c')],
  'children': [{'_id': ObjectId('670d092efcab47f662241035'),
    'first_name': 'Gian',
    'last_name': 'Erard'}],
  'num_children': 1},

### 1c. Check whether flagged "duplicate parents" actually have role == parent

The lookup above joins whatever `_id`s appear in a student's `parents` array without checking `role`. If one of those `_id`s belongs to a `teacher`-role account (same person, different role-account) rather than a second `parent`-role account, it's not really a duplicate-account bug — it's one person with two legitimate role-accounts. This checks the `role` field of every parent_doc involved in a flagged duplicate.

In [16]:
pipeline = [
    {"$lookup": {
        "from": "user",
        "localField": "parents",
        "foreignField": "_id",
        "as": "parent_docs",
    }},
    {"$match": {
        "$expr": {
            "$gt": [
                {"$size": "$parent_docs"},
                {"$size": {"$setUnion": [
                    {"$map": {
                        "input": "$parent_docs",
                        "as": "p",
                        "in": {"$concat": ["$$p.first_name", " ", "$$p.last_name"]},
                    }},
                    [],
                ]}},
            ]
        }
    }},
    {"$project": {
        "first_name": 1,
        "last_name": 1,
        "parent_docs._id": 1,
        "parent_docs.first_name": 1,
        "parent_docs.last_name": 1,
        "parent_docs.role": 1,
    }},
]

df_roles = pd.DataFrame(list(db["student"].aggregate(pipeline)))
df_roles["parent_roles"] = df_roles["parent_docs"].apply(lambda docs: [d.get("role") for d in docs])
df_roles["has_non_parent_role"] = df_roles["parent_roles"].apply(
    lambda roles: any(r != ["parent"] for r in roles)
)

print(f"flagged rows total: {len(df_roles)}")
print(f"flagged rows where a parent_doc role != ['parent']: {df_roles['has_non_parent_role'].sum()}")
df_roles[df_roles["has_non_parent_role"]].head(10)

flagged rows total: 6835
flagged rows where a parent_doc role != ['parent']: 1073


,_id,first_name,last_name,parent_docs,parent_roles,has_non_parent_role
0,5b68a6c0df336100406f4898,Lucas,Nigg,"[{'_id': 5ba549ae4b3a4c00432b851f, 'role': ['p...","[[parent], [parent], [teacher, parent]]",True
2,5b7c31571037ae003b3bb331,Mattia,Andres,"[{'_id': 5b8bbac29fd067002be131b6, 'role': ['p...","[[parent], [teacher, parent], [parent]]",True
3,5b7c313f1037ae003a91d571,Paolo,Andres,"[{'_id': 5b8bbb669fd0670029b093bd, 'role': ['p...","[[parent], [parent], [teacher, parent]]",True
10,5d2585830e3e9e00855fc74d,Timothy,Martin,"[{'_id': 5c1366f5d9f44a00790a6560, 'role': ['p...","[[parent], [parent], [teacher, parent]]",True
11,5d2585840e3e9e00855fc79e,Jonathan,Martin,"[{'_id': 6101691ad330620d24d08f8f, 'last_name'...","[[teacher, parent], [parent], [parent]]",True
23,5ddac4fc6d7b0f0058c2704f,Hebron,Yohannes,"[{'_id': 6681dc54ec2c4673518a5fae, 'role': ['p...","[[parent], [parent], [parent, student]]",True
43,5e709ea79bd54c00445951ab,Sophie,Moor,"[{'_id': 64da12ee3e6d660345847e2c, 'role': ['t...","[[teacher, parent], [parent]]",True
44,5e73004e9b04fd0034e2a725,Erik,Moor,"[{'_id': 64da12ee3e6d660345847e2c, 'role': ['t...","[[teacher, parent], [parent]]",True
73,5e85e6f7095b9d003d7e4989,Erin,Halimi,"[{'_id': 612fc8ae8c09d60d59db2664, 'role': ['p...","[[parent, student], [parent]]",True
76,5e8c42bf2bc725003ccfd523,Lia,Steiner,"[{'_id': 5e95c3e62e81c500355368fe, 'role': ['p...","[[parent], [teacher, parent]]",True


## 2. Parents with the same name as a student

Finds parent docs whose name matches the name of some student doc (independent of whether they're linked as family) — useful to spot mixed-up roles or accidental duplicates.

In [ ]:
pipeline = [
    {"$match": {"user_type": "parent"}},  # TODO: confirm field/value that identifies a parent doc
    {"$lookup": {
        "from": "users",  # TODO: confirm collection name
        "let": {"pname": "$full_name"},  # TODO: parent's name field
        "pipeline": [
            {"$match": {"user_type": "student"}},  # TODO
            {"$match": {"$expr": {"$eq": ["$full_name", "$$pname"]}}},  # TODO: student's name field
        ],
        "as": "matching_students",
    }},
    {"$match": {"matching_students": {"$ne": []}}},
    {"$project": {
        "full_name": 1,
        "matching_students._id": 1,
        "matching_students.full_name": 1,
    }},
]

list(db.users.aggregate(pipeline))

## 3. Users who are both student and parent

Two possible schemas — use whichever matches reality:

- **3a**: a single user doc has a `roles` array containing both values.
- **3b**: role is a separate doc per role, linked by a shared identity field (e.g. email).

In [ ]:
# 3a: single doc, roles stored as an array field
query = {
    "roles": {"$all": ["student", "parent"]},  # TODO: confirm field is "roles" (array), not "user_type" (scalar)
}

list(db.users.find(query))

In [ ]:
# 3b: one doc per role, grouped by a shared identity field (e.g. email / national id)
pipeline = [
    {"$group": {
        "_id": "$email",  # TODO: field that uniquely identifies the *person* across role-docs
        "roles": {"$addToSet": "$user_type"},  # TODO: role field name
        "doc_ids": {"$push": "$_id"},
    }},
    {"$match": {"roles": {"$all": ["student", "parent"]}}},
]

list(db.users.aggregate(pipeline))